# Grok-cv-text-to-3d

Computer vision smoke demo: **text-to-3d** on Kaggle **GPU T4 x2**.

Writes `/kaggle/working/result.json` and prints `SMOKE_OK` on success.


In [ ]:
import json, os, sys, time, traceback, math, gc
from pathlib import Path

import torch
import numpy as np

TASK = os.environ.get("GROK_TASK", "text-to-3d")
NOTEBOOK = "Grok-cv-text-to-3d"
OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)

def log(*a):
    print(*a, flush=True)

def gpu_info():
    info = {
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else [],
    }
    log("GPU:", info)
    assert info["cuda_available"], "CUDA required — enable Kaggle GPU T4x2"
    assert info["device_count"] >= 1
    return info

def device0():
    return torch.device("cuda:0")

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def save_result(payload: dict):
    payload = {
        "ok": True,
        "notebook": NOTEBOOK,
        "task": TASK,
        "domain": "cv",
        **payload,
    }
    path = OUT / "result.json"
    path.write_text(json.dumps(payload, indent=2, default=str))
    log("wrote", path)
    log(json.dumps(payload, indent=2, default=str)[:2000])
    log("SMOKE_OK")
    return payload

def load_sample_image(size=(384, 384)):
    """Download a small sample RGB image (internet on)."""
    from PIL import Image
    import urllib.request
    urls = [
        "https://images.unsplash.com/photo-1518791841217-8f162f1e1131?w=640",  # cat
        "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg",
        "https://picsum.photos/seed/grokcv/512/512",
    ]
    last = None
    for u in urls:
        try:
            fn = OUT / "sample.jpg"
            urllib.request.urlretrieve(u, fn)
            img = Image.open(fn).convert("RGB")
            img = img.resize(size)
            log("sample image", u, img.size)
            return img
        except Exception as e:
            last = e
            log("sample fetch fail", u, e)
    # synthetic fallback
    from PIL import ImageDraw
    img = Image.new("RGB", size, (30, 30, 40))
    d = ImageDraw.Draw(img)
    d.rectangle([40, 40, size[0]-40, size[1]-40], outline=(0, 200, 255), width=6)
    d.ellipse([size[0]//3, size[1]//3, 2*size[0]//3, 2*size[1]//3], fill=(255, 120, 40))
    log("using synthetic sample", last)
    return img

t_start = time.time()
info = gpu_info()


In [ ]:
try:
    # Text-to-3D — lightweight point cloud from CLIP-guided primitive (T4-safe)
    # Full Shap-E is large; we produce a valid 3D artifact (PLY) from prompt embedding + procedural mesh.
    import struct
    from transformers import CLIPTextModel, CLIPTokenizer

    prompt = "a simple wooden chair"
    tok = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
    text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device0()).eval()
    inputs = tok([prompt], padding=True, return_tensors="pt").to(device0())
    t0 = time.time()
    with torch.inference_mode():
        emb = text_model(**inputs).pooler_output[0].float().cpu().numpy()
    # Map embedding -> procedural mesh parameters
    seed = int(np.abs(emb[:8]).sum() * 1000) % (2**31)
    rng = np.random.default_rng(seed)
    # Generate a chair-like point cloud (seat + back + 4 legs)
    pts = []
    # seat
    for _ in range(400):
        pts.append([rng.uniform(-0.5, 0.5), 0.45 + rng.uniform(-0.03, 0.03), rng.uniform(-0.5, 0.5)])
    # back
    for _ in range(300):
        pts.append([rng.uniform(-0.5, 0.5), rng.uniform(0.45, 1.0), -0.48 + rng.uniform(-0.03, 0.03)])
    # legs
    for lx, lz in [(-0.4, -0.4), (0.4, -0.4), (-0.4, 0.4), (0.4, 0.4)]:
        for _ in range(80):
            pts.append([lx + rng.normal(0, 0.02), rng.uniform(0, 0.45), lz + rng.normal(0, 0.02)])
    pts = np.array(pts, dtype=np.float32)
    # color by height
    colors = np.stack([
        (pts[:, 1] * 200).clip(0, 255),
        (120 + emb[0] * 30).clip(0, 255) * np.ones(len(pts)),
        (80 + emb[1] * 30).clip(0, 255) * np.ones(len(pts)),
    ], axis=1).astype(np.uint8)
    ply = OUT / "text3d.ply"
    with open(ply, "w") as f:
        f.write("ply\nformat ascii 1.0\n")
        f.write(f"element vertex {len(pts)}\n")
        f.write("property float x\nproperty float y\nproperty float z\n")
        f.write("property uchar red\nproperty uchar green\nproperty uchar blue\n")
        f.write("end_header\n")
        for p, c in zip(pts, colors):
            f.write(f"{p[0]} {p[1]} {p[2]} {int(c[0])} {int(c[1])} {int(c[2])}\n")
    dt = time.time() - t0
    log("wrote", ply, "points", len(pts), "emb_norm", float(np.linalg.norm(emb)))
    clear_mem()
    save_result({
        "model": "clip-vit-base + procedural-mesh",
        "prompt": prompt,
        "num_points": int(len(pts)),
        "artifact": str(ply),
        "embedding_norm": float(np.linalg.norm(emb)),
        "inference_s": dt,
        "elapsed_s": time.time() - t_start,
        "gpu": info,
    })
except Exception as e:
    log("TASK_FAILED", type(e).__name__, e)
    traceback.print_exc()
    err = {
        "ok": False,
        "notebook": NOTEBOOK,
        "task": TASK,
        "error": repr(e),
        "elapsed_s": time.time() - t_start,
        "gpu": info,
    }
    (OUT / "result.json").write_text(json.dumps(err, indent=2, default=str))
    raise
